# SPAR — 4 composites x 7 tiles, gross/net toggle, as-of 0CQ

Fabric **Python** notebook (not PySpark). Pulls SPAR Engine results for four GIPS
composite ACCTs against each one's default benchmark, across seven saved SPAR components
("tiles"), on a gross / net / both basis, and lands them in `hbcm_datahub`.

```
grid = TILES x STRATEGIES x FEE_BASIS  ->  one SPAR calculation unit each
     = 7     x 4          x 2          =  56 units, submitted as 7 per-tile batches
```

## The tiles

`enddate` is **`0CQ`** — most recent calendar quarter end — for every tile. Only
`startdate` varies.

| Tile | Start | Peer universe |
|---|---|---|
| `multi_horizon_returns` | inception | — |
| `calendar_year_returns` | inception | — |
| `cumulative_monthly` | inception | — |
| `monthly_raw_returns` | inception | — |
| `peer_multi_horizon` | inception | **yes** |
| `peer_calendar_year` | inception | **yes** |
| `risk_stats_3y` | −3Y | — |
| `risk_stats_itd` *(planned)* | inception | — |

Risk stats want a fixed 3-year window while the cumulative, calendar-year, multi-horizon
and monthly tiles want inception-to-date, so `startdate` is per tile.
`startdate: "INCEPTION"` resolves per-strategy from that ACCT's inception date, which is
how the planned since-inception risk variant reuses the same component id under a second
tile key.

`universeid` is sent **only** for the two peer tiles. On the others it is omitted from
the payload entirely rather than sent as null, which would override the component's
saved universe with nothing.

## Dates

`0CQ` uses FactSet's calendar-period grammar (`CQ` / `CY` for calendar, `FQ` / `FY` for
fiscal), so it's unambiguous where a bare `0Q` is not. SPAR Engine has no `DatesApi`, so
relative tokens resolve server-side only and can't be verified before submitting.

`AS_OF_ABS` — the most recent completed calendar quarter end, computed in plain Python —
always labels the output, so landed rows carry a real date rather than a token. Cell 3
prints both; if they ever disagree the labels are wrong. Set
`USE_ABSOLUTE_AS_OF = True` to send the absolute date instead of `0CQ`, which is what you
want for a backfill or restatement re-run where the request should be identical on every
replay rather than drifting as quarters roll.

## Fee basis

Each ACCT carries **both** a gross and a net return stream, so the toggle is
`SPARIdentifier.returntype` — one account id, two return types. Set
`SYMBOL_SUFFIX_MODE = True` for the alternative QR/`$$Performance.ofdb` layout where
gross and net are separate `_G` / `_N` symbols.

Return type values are often numeric codes, not the literal strings `"Gross"` / `"Net"`.
Cell 4a prints what the API reports per ACCT — take the values from there.

## What you must fill in before first run

All of it is workstation-sourced, all in Cell 3, and Cell 4 discovers every piece:

| Field | Discovery |
|---|---|
| `STRATEGIES[code]["acct"]` + `["prefix"]` | Cell 4a |
| `STRATEGIES[code]["returntype"]` (gross, net) | Cell 4a |
| `STRATEGIES[code]["benchmark"]` (id, prefix) | Cell 4e — **a wrong prefix returns a bare 400 with no message** |
| `STRATEGIES[code]["inception"]` | from the composite record |
| `STRATEGIES[code]["universe"]` | Cell 4c |
| `TILES[tile]["componentid"]` | Cell 4b |
| `TILES[tile]["frequency"]` | Cell 4d |

Cell 3 asserts that every peer tile has a universe and every `INCEPTION` tile has an
inception date, so bad config fails before an API call is spent.

## Versions — latest as of 2026-08-06

| Package | Version | Notes |
|---|---|---|
| `fds.sdk.SPAREngine` | **3.0.0** | released 2026-05-20 |
| `fds.sdk.utils` | 3.0.1 | OAuth helper |
| `fds.protobuf.stach.extensions` | 1.3.3 | STACH parsing |
| `deltalake` | 1.6.2 | OneLake write |

SPAREngine 3.0.0 is a **major bump for runtime hygiene, not an API redesign**. Per
FactSet's `BREAKING.md` (2026-05-20) every Python SDK was bumped together: Python
3.7/3.8/3.9 support dropped, and `urllib3` moved from `>=1.25.3,<2.1.0` to `>=2.7.0`.
Every model and method this notebook touches is unchanged between the 2.x docs and 3.0.0.

Minimum Python is now **3.10**. Fabric Python notebook kernels are 3.10 / 3.11 / 3.12
with **3.12 the default**, so any current kernel satisfies it — prefer 3.12, since 3.10
reaches end of support in October 2026.

> ⚠️ The SPAR SDK vendored under `code/python/SPAREngine/v3/` **in this repo is 2.0.3**
> (last synced 2025-07-21) and predates both the `universeid` field and
> `SPARPeerUniverseApi`. Verify any SPAR question against upstream `main`, not the local
> mirror. PA Engine is at **4.0.0** upstream and carries a second, separate breaking
> change (2026-07-21: required fields dropped from `PADateParameters`) that matters for
> the PA side of the pipeline but not for this notebook.

## Library setup — read this before scheduling

Do **not** rely on `%pip install` here. Per Microsoft, inline installs are *disabled by
default in pipeline runs* and *unsupported in reference runs*, and `%pip`-installed
libraries are not retained across runs. Attach the four packages above to a **Fabric
Environment** and bind this notebook to it.

For interactive first-run only:
```
%pip install fds.sdk.SPAREngine==3.0.0 fds.sdk.utils==3.0.1 \
             fds.protobuf.stach.extensions==1.3.3 deltalake==1.6.2
```

In [ ]:
# === Cell 1: credentials ===================================================
# HBCM_Config defines FACTSET_USER and FACTSET_APIKEY as plain strings.
# %run works in both interactive and pipeline mode; notebookutils.notebook.run() does not
# propagate variables, so keep this as %run.
%run HBCM_Config

In [ ]:
# === Cell 2: imports + API client ==========================================
import json, time, datetime as dt
import pandas as pd

import fds.sdk.SPAREngine
from fds.sdk.SPAREngine.api import (
    spar_calculations_api,
    spar_peer_universe_api,   # SDK >= 2.1; absent from the 2.0.3 copy vendored here
    accounts_api,
    components_api,
    benchmarks_api,
    frequencies_api,
)
from fds.sdk.SPAREngine.models import (
    SPARCalculationParametersRoot,
    SPARCalculationParameters,
    SPARIdentifier,
    SPARDateParameters,
    CalculationMeta,
)
from urllib3 import Retry   # SDK 3.0.0 requires urllib3 >= 2.7.0

# Fail loudly on a stale Environment rather than 400-ing later on universeid.
# Compare the major as an int — a string compare would rank "10.0.0" below "3.0.0".
SDK_VERSION = fds.sdk.SPAREngine.__version__
assert int(SDK_VERSION.split(".")[0]) >= 3, (
    f"fds.sdk.SPAREngine {SDK_VERSION} found; this notebook targets >=3.0.0. "
    "Check the bound Fabric Environment."
)
print("SPAREngine SDK", SDK_VERSION)

configuration = fds.sdk.SPAREngine.Configuration(
    username=FACTSET_USER,
    password=FACTSET_APIKEY,
)
# Preferred once an app-config.json exists in Key Vault:
#   from fds.sdk.utils.authentication import ConfidentialClient
#   configuration = fds.sdk.SPAREngine.Configuration(
#       fds_oauth_client=ConfidentialClient(str(config_path)))

# Retry server-side failures only. Never add 429 with a naive backoff — urllib3 already
# honours Retry-After for 429, and 4xx other than 429 will not fix themselves.
configuration.retries = Retry(
    total=3,
    status_forcelist=[500, 502, 503, 504],
    backoff_factor=2,
    allowed_methods=frozenset(["GET", "POST"]),
)

api_client = fds.sdk.SPAREngine.ApiClient(configuration)
calc_api = spar_calculations_api.SPARCalculationsApi(api_client)

In [ ]:
# === Cell 3: THE CONFIG BLOCK — the only cell you normally edit ============

CURRENCY = "USD"            # all GIPS series are USD-denominated

# --- as-of date (uniform across every tile) --------------------------------
# "0CQ" = most recent calendar quarter end. FactSet's relative-date grammar uses CQ /
# CY for calendar periods and FQ / FY for fiscal, so 0CQ is unambiguous where a bare
# "0Q" is not. Only `startdate` varies per tile; `enddate` is global.
#
# SPAR Engine has no DatesApi, so relative tokens are resolved server-side only — they
# cannot be verified client-side before submitting.
AS_OF_RELATIVE = "0CQ"

# Flip to True to send the absolute date instead. Relative is right for the normal
# "latest quarter" run; absolute is right for a backfill or a restatement re-run, where
# you want the request to be byte-identical on every replay rather than drifting as
# quarters roll over.
USE_ABSOLUTE_AS_OF = False

def _prior_quarter_end(today=None):
    """Most recent COMPLETED calendar quarter end — the absolute equivalent of 0CQ."""
    d = today or dt.date.today()
    qe = dt.date(d.year, ((d.month - 1) // 3) * 3 + 1, 1) - dt.timedelta(days=1)
    return qe.strftime("%Y%m%d")

AS_OF_ABS = _prior_quarter_end()
END_DATE = AS_OF_ABS if USE_ABSOLUTE_AS_OF else AS_OF_RELATIVE

# AS_OF_ABS always labels the output, even when the request used the relative token, so
# the landed rows carry a real date. If SPAR resolves 0CQ to anything other than
# AS_OF_ABS the labels would be wrong — print both and eyeball them on first run.
print(f"enddate sent: {END_DATE!r}   asof_date written: {AS_OF_ABS}")

# --- fee basis toggle ------------------------------------------------------
# Each ACCT carries BOTH a gross and a net return stream, so the toggle is
# SPARIdentifier.returntype — one account id, two return types. (Contrast the QR/OFDB
# layout where gross and net are separate GIPS_<STRAT>_G / _N symbols; that path is
# still available via SYMBOL_SUFFIX_MODE below.)
#
# returntype values go in STRATEGIES[...]["returntype"]. They are often numeric codes
# rather than the literal strings "Gross"/"Net" — Cell 4a prints what the API reports.
FEE_BASIS = "both"          # "gross" | "net" | "both"
SYMBOL_SUFFIX_MODE = False  # True => append _G/_N to the account id instead
SUFFIX = {"gross": "_G", "net": "_N"}

# --- benchmark + peer universe groups --------------------------------------
# LC and LCS share the Russell 1000 benchmark AND the same peer universe, so that pair
# is defined once here and referenced by both. One source of truth, so the two can never
# drift apart in config.
#
# This is a config-level share, NOT a request-level one. See the note on `bench_group`
# in STRATEGIES below for why the two strategies still get separate calculation units.
# Three distinct benchmarks across four composites:
#   r1000  Russell 1000  -> LC, LCS   (the shared pair)
#   r2500  Russell 2500  -> SMID
#   r3000  Russell 3000  -> CONC
# Only the ids and prefixes are still TODO. Confirm each with BenchmarksApi (Cell 4e)
# before the first real run — a wrong prefix is a bare 400 with no diagnostic.
BENCHMARK_GROUPS = {
    "r1000": {   # Russell 1000
        "benchmark": {"id": "<TODO>", "prefix": "RUSSELL:", "returntype": None},
        "universe":  "<TODO shared LC/LCS peer universe>",
    },
    "r2500": {   # Russell 2500
        "benchmark": {"id": "<TODO>", "prefix": "RUSSELL:", "returntype": None},
        "universe":  "<TODO>",
    },
    "r3000": {   # Russell 3000
        "benchmark": {"id": "<TODO>", "prefix": "RUSSELL:", "returntype": None},
        "universe":  "<TODO>",
    },
}

# --- the four composites ---------------------------------------------------
# TODO — acct, prefix, returntype and inception per ACCT. Cell 4 discovers all of them.
#
# `bench_group` points at BENCHMARK_GROUPS above. LC and LCS both point at "r1000";
# SMID is Russell 2500 and CONC is Russell 3000, so LC + LCS are confirmed as the pair.
#
# Sharing a benchmark does NOT mean sharing a calculation unit. SPAR takes one `dates`
# object per unit, so two strategies in one unit must share a start date — which is
# incompatible with resolving INCEPTION per strategy. Explicit per-strategy inception is
# worth more than saving a handful of units, so each strategy keeps its own unit.
STRATEGIES = {
    "LC": {
        "label":       "Large Cap",
        "acct":        "<TODO>",
        "prefix":      "CLIENT:",
        "returntype":  {"gross": "<TODO>", "net": "<TODO>"},
        "inception":   "<TODO YYYYMMDD>",
        "bench_group": "r1000",       # Russell 1000, shared with LCS
    },
    "SMID": {
        "label":       "SMID",
        "acct":        "<TODO>",
        "prefix":      "CLIENT:",
        "returntype":  {"gross": "<TODO>", "net": "<TODO>"},
        "inception":   "<TODO YYYYMMDD>",
        "bench_group": "r2500",       # Russell 2500
    },
    "LCS": {
        "label":       "Large Cap Select",
        "acct":        "<TODO>",
        "prefix":      "CLIENT:",
        "returntype":  {"gross": "<TODO>", "net": "<TODO>"},
        "inception":   "<TODO YYYYMMDD>",
        "bench_group": "r1000",          # shares with LC
    },
    "CONC": {
        "label":       "Concentrated Equity",
        "acct":        "<TODO>",
        "prefix":      "CLIENT:",
        "returntype":  {"gross": "<TODO>", "net": "<TODO>"},
        "inception":   "<TODO YYYYMMDD>",
        "bench_group": "r3000",       # Russell 3000
    },
}

def strategy_config(code: str) -> dict:
    """Strategy record with its benchmark group flattened in."""
    s = dict(STRATEGIES[code])
    s.update(BENCHMARK_GROUPS[s["bench_group"]])
    return s

# --- the tiles -------------------------------------------------------------
# A "tile" is a saved SPAR component. It fixes the statistic columns server-side; the
# POST body acts as a one-time override of the component's saved dates.
#
#   startdate       "INCEPTION" resolves per-strategy from STRATEGIES[...]["inception"],
#                   so every gross/net pair of a strategy starts at that strategy's own
#                   inception. Everything that can use it does; the risk tile uses the
#                   relative -3Y window instead. enddate is END_DATE for every tile.
#   needs_universe  True  -> universeid is sent, and the group must have one.
#                   False -> universeid omitted entirely (never sent as null).
#   frequency       Confirm against FrequenciesApi (Cell 4d).
#
# A component id may appear more than once under different variant keys — that is how
# the eventual "risk stats since inception" variant sits alongside the 3Y one.
#   component_name  The component's name AS SAVED IN THE WORKSTATION. This is the
#                   contract, not the id — Cell 4 resolves name -> id on every run,
#                   because re-saving a component can mint a new id and a stale
#                   hardcoded id 400s with no hint that the id is what broke.
#   pinned_componentid  Optional. Last known id. Purely for drift detection: if the
#                   resolved id differs, Cell 4 shouts, because a re-saved component may
#                   also have had its columns changed. Leave None on first run, then
#                   paste in what Cell 4 printed.
#   time_series     True for tiles whose output is a dated series rather than one row
#                   per period-label. Used for the ragged-series report in Cell 8.
TILES = {
    "multi_horizon_returns": {
        "component_name": "<TODO exact workstation name>", "pinned_componentid": None,
        "startdate": "INCEPTION", "frequency": "Monthly",
        "needs_universe": False, "time_series": False,
    },
    "calendar_year_returns": {
        "component_name": "<TODO exact workstation name>", "pinned_componentid": None,
        "startdate": "INCEPTION", "frequency": "Monthly",
        "needs_universe": False, "time_series": False,
    },
    "cumulative_monthly": {
        "component_name": "<TODO exact workstation name>", "pinned_componentid": None,
        "startdate": "INCEPTION", "frequency": "Monthly",
        "needs_universe": False, "time_series": True,
    },
    "monthly_raw_returns": {
        "component_name": "<TODO exact workstation name>", "pinned_componentid": None,
        "startdate": "INCEPTION", "frequency": "Monthly",
        "needs_universe": False, "time_series": True,
    },
    "peer_multi_horizon": {
        "component_name": "<TODO exact workstation name>", "pinned_componentid": None,
        "startdate": "INCEPTION", "frequency": "Monthly",
        "needs_universe": True, "time_series": False,
    },
    "peer_calendar_year": {
        "component_name": "<TODO exact workstation name>", "pinned_componentid": None,
        "startdate": "INCEPTION", "frequency": "Monthly",
        "needs_universe": True, "time_series": False,
    },
    "risk_stats_3y": {
        "component_name": "<TODO exact workstation name>", "pinned_componentid": None,
        "startdate": "-3Y", "frequency": "Monthly",      # relative, not inception
        "needs_universe": False, "time_series": False,
    },
    # Planned variant — SAME component_name as risk_stats_3y, inception-to-date window.
    # Name-based resolution means both keys resolve to the same live id automatically.
    # "risk_stats_itd": {
    #     "component_name": "<same name as risk_stats_3y>", "pinned_componentid": None,
    #     "startdate": "INCEPTION", "frequency": "Monthly",
    #     "needs_universe": False, "time_series": False,
    # },
}

SPAR_DOCUMENT = "Client:/SPAR/HBCM"    # searched for component names every run
PEER_UNIVERSE_CATEGORY = "Custom"      # for the peer universe discovery cell only

# --- execution -------------------------------------------------------------
# 7 tiles x 4 strategies x 2 bases = 56 units. Submitting all of them as one calculation
# is all-or-nothing and pushes the long-running window; batching per tile gives 8 units
# per calc, isolates failures to one tile, and stays clear of the ~5-10 concurrent-calc
# ceiling. Set False only for a small ad hoc run.
BATCH_PER_TILE = True

# --- OneLake target --------------------------------------------------------
WORKSPACE_ID = "1b9fac18-9d75-4437-ab6c-b6ba44ff46a8"   # HBCM - Production
LAKEHOUSE_ID = "7cdf13b1-4586-4a02-b8ff-72fcf6db1277"   # hbcm_datahub (schema-enabled)
ONELAKE = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}"
TABLE_PATH = f"{ONELAKE}/Tables/factset/spar_composite_returns"
RAW_DIR = f"{ONELAKE}/Files/raw/spar"

BASES = ["gross", "net"] if FEE_BASIS == "both" else [FEE_BASIS]

# Fail fast on config that cannot possibly work, before burning an API call.
for _c, _s in STRATEGIES.items():
    assert _s["bench_group"] in BENCHMARK_GROUPS, \
        f"{_c}: unknown bench_group {_s['bench_group']!r}"
for _t, _cfg in TILES.items():
    if _cfg["needs_universe"]:
        _bad = [c for c in STRATEGIES if not strategy_config(c).get("universe")]
        assert not _bad, f"tile {_t} needs a universe; missing for {_bad}"
    if _cfg["startdate"] == "INCEPTION":
        _bad = [c for c, s in STRATEGIES.items() if not s.get("inception")]
        assert not _bad, f"tile {_t} starts at INCEPTION; missing for {_bad}"

print(f"{len(TILES)} tiles x {len(STRATEGIES)} strategies x {len(BASES)} basis "
      f"= {len(TILES) * len(STRATEGIES) * len(BASES)} units"
      f"{f' in {len(TILES)} batches' if BATCH_PER_TILE else ' in 1 calculation'}")

In [ ]:
# === Cell 4: resolve component ids BY NAME — every run ====================
# Component ids are not stable. Editing and re-saving a SPAR component in the workstation
# can mint a new id, and a stale hardcoded id fails as a 400 with no hint that the id is
# the problem. So the tile's workstation NAME is the contract, and the id is looked up
# fresh on every run. This is pipeline code, not discovery.

comp_api = components_api.ComponentsApi(api_client)

def _field(obj, name):
    """SDK models allow attribute or dict-style access depending on construction."""
    if hasattr(obj, name):
        return getattr(obj, name)
    try:
        return obj.get(name)
    except AttributeError:
        return None

def resolve_component_ids(document: str = SPAR_DOCUMENT):
    """tile name -> live component id, plus any drift against pinned ids."""
    summary = comp_api.get_spar_components(document=document)

    by_name = {}
    for cid, meta in (summary.data or {}).items():
        by_name.setdefault(_field(meta, "name"), []).append(cid)

    resolved, drift, missing = {}, [], []
    for tile, cfg in TILES.items():
        name = cfg["component_name"]
        hits = by_name.get(name, [])
        if len(hits) != 1:
            # Ambiguous or absent — refuse to guess which component was meant.
            missing.append((tile, name, len(hits)))
            continue
        cid = hits[0]
        resolved[tile] = cid
        pinned = cfg.get("pinned_componentid")
        if pinned and pinned != cid:
            drift.append((tile, name, pinned, cid))

    return resolved, drift, missing, by_name

RESOLVED_COMPONENTS, COMPONENT_DRIFT, COMPONENT_MISSING, COMPONENTS_BY_NAME = \
    resolve_component_ids()

for tile, cid in RESOLVED_COMPONENTS.items():
    print(f"{tile:<24} {cid}  ({TILES[tile]['component_name']})")

if COMPONENT_DRIFT:
    # Not fatal — a re-saved component is normal. But it must be loud, because it means
    # the component may have been redefined, so the columns could have moved too.
    print("\n*** COMPONENT ID DRIFT — verify the component still returns what you expect,")
    print("*** then update pinned_componentid in Cell 3:")
    for tile, name, was, now in COMPONENT_DRIFT:
        print(f"    {tile}: {name!r}  {was} -> {now}")

if COMPONENT_MISSING:
    print("\nUnresolved tiles (name absent from the document, or not unique):")
    for tile, name, n in COMPONENT_MISSING:
        print(f"    {tile}: {name!r} matched {n} components")
    print("\nComponent names available in", SPAR_DOCUMENT)
    for name, cids in sorted(COMPONENTS_BY_NAME.items(), key=lambda kv: str(kv[0])):
        print(f"    {name!r}: {cids}")

assert not COMPONENT_MISSING, "every tile needs exactly one matching component name"

In [ ]:
# === Cell 4b: DISCOVERY — run interactively once, then leave it alone ======
# Resolves the TODOs in Cell 3. Not part of the scheduled path.

acct_api = accounts_api.AccountsApi(api_client)
bmk_api = benchmarks_api.BenchmarksApi(api_client)   # comp_api is created in Cell 4
peer_api = spar_peer_universe_api.SPARPeerUniverseApi(api_client)
freq_api = frequencies_api.FrequenciesApi(api_client)

# (a) Return types per ACCT -> STRATEGIES[...]["returntype"]
#     This is the authoritative answer on gross vs net. ReturnType carries (name, id);
#     the `id` is what goes in SPARIdentifier.returntype and is often a numeric code,
#     not the literal string "Net".
for code, s in STRATEGIES.items():
    path = s["acct"] if s["acct"].endswith(".ACCT") else f"{s['acct']}.ACCT"
    try:
        resp = acct_api.get_spar_returns_type(path)   # URL-encoded account path
        types = [(rt.get("name"), rt.get("id")) for rt in (resp.data.returns_type or [])]
        print(f"OK   {code:8s} {path:36s} -> {types}")
    except fds.sdk.SPAREngine.ApiException as e:
        print(f"FAIL {code:8s} {path:36s} -> {e.status} {e.body}")

# (b) Component names -> TILES[...]["component_name"]. Cell 4 already resolves these on
#     every run and prints the full name list if any tile fails to match, so there is
#     nothing to do here. Kept as a pointer.

# (c) Peer universes -> STRATEGIES[...]["universe"] (needed by the two peer tiles).
#     `category` is required; `name` and `directory` are optional filters.
try:
    print(peer_api.get_list_of_peer_universe(PEER_UNIVERSE_CATEGORY))
except fds.sdk.SPAREngine.ApiException as e:
    print(f"peer universe lookup failed: {e.status} {e.body}")

# (d) Valid frequency ids -> TILES[...]["frequency"]. Calendar-year and multi-horizon
#     tiles may want something other than Monthly.
try:
    print(freq_api.get_spar_frequencies())
except fds.sdk.SPAREngine.ApiException as e:
    print(f"frequencies lookup failed: {e.status} {e.body}")

# (e) Confirm a benchmark id resolves and see its valid prefixes -> ["benchmark"]
#   print(bmk_api.get_spar_benchmark_by_id(id="<benchmark id>"))

In [ ]:
# === Cell 5: build the calculation units ==================================
# One unit per (tile, strategy, basis) = 7 x 4 x 2 = 56.
#
# LC and LCS share a benchmark and peer universe, so it is tempting to put both in one
# unit's `accounts` list. Don't: SPARCalculationParameters carries ONE `dates` object per
# unit, so accounts sharing a unit share a start date — which defeats resolving INCEPTION
# per strategy. (SPAR does expose `useeachportfolioinception` for exactly that case, but
# it hands the window to the engine, so the request no longer states what it computed and
# there is nothing to verify against. Explicit beats implicit here.) The shared benchmark
# is deduplicated in config instead, via BENCHMARK_GROUPS.

def account_identifier(code: str, basis: str) -> SPARIdentifier:
    s = strategy_config(code)
    if SYMBOL_SUFFIX_MODE:
        # Separate gross/net symbols (QR / $$Performance.ofdb layout).
        return SPARIdentifier(id=f"{s['acct']}{SUFFIX[basis]}", returntype=None,
                              prefix=s["prefix"])
    # One ACCT carrying both streams — pick the stream by return type.
    return SPARIdentifier(id=s["acct"], returntype=s["returntype"][basis],
                          prefix=s["prefix"])

def resolve_startdate(tile_cfg: dict, code: str) -> str:
    """INCEPTION -> that strategy's own inception date; anything else passes through.

    Resolution is per strategy, so LC's gross and net units both start at LC's inception
    and SMID's both start at SMID's — no cross-contamination between strategies, and no
    single global start date.
    """
    sd = tile_cfg["startdate"]
    return STRATEGIES[code]["inception"] if sd == "INCEPTION" else sd

def build_unit(tile_name: str, tile_cfg: dict, code: str, basis: str) -> SPARCalculationParameters:
    s = strategy_config(code)
    bmk = s["benchmark"]
    kwargs = dict(
        componentid=RESOLVED_COMPONENTS[tile_name],   # resolved by name in Cell 4
        accounts=[account_identifier(code, basis)],
        benchmark=SPARIdentifier(id=bmk["id"], returntype=bmk["returntype"],
                                 prefix=bmk["prefix"]),
        dates=SPARDateParameters(
            startdate=resolve_startdate(tile_cfg, code),
            enddate=END_DATE,          # 0CQ (or its absolute equivalent) for every tile
            frequency=tile_cfg["frequency"],
            # False deliberately: the per-strategy inception is already resolved above
            # and sent explicitly, so each unit's window is visible in the request.
            # Setting True would delegate it to the engine and make it unverifiable.
            useeachportfolioinception=False,
        ),
        currencyisocode=CURRENCY,
    )
    # Send universeid only for the peer tiles. Passing None on a non-peer tile would
    # override the component's saved universe with nothing.
    if tile_cfg["needs_universe"]:
        kwargs["universeid"] = s["universe"]
    return SPARCalculationParameters(**kwargs)

def unit_key(tile_name: str, code: str, basis: str) -> str:
    return f"{tile_name}__{code}__{basis}"

def make_root(units: dict) -> SPARCalculationParametersRoot:
    return SPARCalculationParametersRoot(
        data=units,
        meta=CalculationMeta(
            contentorganization="SimplifiedRow",
            stach_content_organization="SimplifiedRow",
            contenttype="Json",
            format="JsonStach",
        ),
    )

UNIT_KEYS = {}          # unit_key -> (tile_name, strategy_code, basis)
batches = []            # list[(batch_label, SPARCalculationParametersRoot)]
all_units = {}

for tile_name, tile_cfg in TILES.items():
    tile_units = {}
    for code in STRATEGIES:
        for basis in BASES:
            key = unit_key(tile_name, code, basis)
            tile_units[key] = build_unit(tile_name, tile_cfg, code, basis)
            UNIT_KEYS[key] = (tile_name, code, basis)
    all_units.update(tile_units)
    if BATCH_PER_TILE:
        batches.append((tile_name, make_root(tile_units)))

if not BATCH_PER_TILE:
    batches = [("all", make_root(all_units))]

# Show the resolved window per tile x strategy. Read down a column: the inception tiles
# must show that strategy's own date, and only risk_stats should show -3Y.
print(f"{len(all_units)} units in {len(batches)} batch(es)\n")
print("resolved startdate -> enddate " + repr(END_DATE) + "\n")
print(f"{'tile':<24}" + "".join(f"{c:<12}" for c in STRATEGIES))
for tile_name, tile_cfg in TILES.items():
    row = "".join(f"{resolve_startdate(tile_cfg, c):<12}" for c in STRATEGIES)
    print(f"{tile_name:<24}{row}")
print()
for code in STRATEGIES:
    s = strategy_config(code)
    print(f"{code:<6} bench={s['benchmark']['prefix']}{s['benchmark']['id']:<24} "
          f"universe={s['universe']:<20} group={s['bench_group']}")

In [ ]:
# === Cell 6: submit + poll ================================================
# Multi-unit calculations ALWAYS return 202 regardless of the deadline header, so the
# polling loop is the normal path here, not the exception.

def run_spar(params_root, deadline=10, poll_interval=3, timeout=900):
    """Returns list[(unit_id, result_or_None, status)] for one calculation."""
    wrapper = calc_api.post_and_calculate(
        x_fact_set_api_long_running_deadline=deadline,
        spar_calculation_parameters_root=params_root,
    )
    code = wrapper.get_status_code()
    if code == 200:
        status_root = wrapper.get_response_200()
    elif code == 201:
        status_root = wrapper.get_response_201()
    elif code == 202:
        status_root = wrapper.get_response_202()
        calc_id = status_root.data.calculationid
        deadline_at = time.time() + timeout
        while True:
            if time.time() > deadline_at:
                calc_api.cancel_calculation_by_id(id=calc_id)
                raise TimeoutError(f"calc {calc_id} exceeded {timeout}s (cancelled)")
            poll = calc_api.get_calculation_status_by_id(id=calc_id)
            if poll.get_status_code() == 200:
                status_root = poll.get_response_200()
                break
            if poll.get_status_code() != 202:
                raise RuntimeError(f"unexpected poll status {poll.get_status_code()}")
            time.sleep(poll_interval)
    else:
        raise RuntimeError(f"unexpected submit status {code}")

    calc_id = status_root.data.calculationid
    out = []
    for unit_id, unit_status in (status_root.data.units or {}).items():
        st = getattr(unit_status, "status", None)
        if st != "Success":
            out.append((unit_id, None, st))
            continue
        res = calc_api.get_calculation_unit_result_by_id(id=calc_id, unit_id=unit_id)
        out.append((unit_id, res, st))
    return calc_id, out

results, calc_ids, batch_errors = [], {}, {}
for label, root in batches:
    try:
        cid, batch_results = run_spar(root)
        calc_ids[label] = cid
        results.extend(batch_results)
        n_bad = sum(1 for _, r, _ in batch_results if r is None)
        print(f"{label:24s} calc={cid} ok={len(batch_results) - n_bad} failed={n_bad}")
    except Exception as e:
        # One bad tile should not cost the other six. Record and continue.
        batch_errors[label] = repr(e)
        print(f"{label:24s} BATCH FAILED: {e!r}")

failed = [(u, s) for u, r, s in results if r is None]
for u, s in failed:
    print(f"  FAILED unit {u}: {s}")

# Log calc ids alongside X-DataDirect-Request-Key for any FactSet support ticket —
# use the *_with_http_info variants when you need the response headers.
print(f"\n{len(results) - len(failed)} usable units; "
      f"{len(failed)} failed units; {len(batch_errors)} failed batches")
assert results and not failed and not batch_errors, (
    "resolve failures before writing to the lakehouse — a partial snapshot is worse "
    "than none, because it looks complete downstream"
)

In [ ]:
# === Cell 7: raw landing (write-once audit copy) ==========================
# FactSet earns a raw layer: calls are slow and async, STACH reshaping is fiddly, and
# vendor restatements mean the same call replayed later does NOT return what it
# originally returned. That matters for GIPS and Marketing Rule substantiation.
# Files, not Delta — raw stays invisible to the SQL endpoint, which is correct.

asof_tag = AS_OF_ABS   # label raw by resolved quarter end, never by a relative token
for unit_id, res, _ in results:
    notebookutils.fs.put(
        f"{RAW_DIR}/asof={asof_tag}/{unit_id}.json",
        json.dumps(res.to_dict(), default=str),
        True,
    )
print(f"landed {len(results)} raw payloads under {RAW_DIR}/asof={asof_tag}/")

In [ ]:
# === Cell 8: STACH -> tidy DataFrame ======================================
from fds.protobuf.stach.extensions.StachExtensionFactory import StachExtensionFactory
from fds.protobuf.stach.extensions.StachVersion import StachVersion

def stach_to_dataframes(api_response):
    """STACH v2 payload -> list[DataFrame], one per table."""
    ext = StachExtensionFactory.get_stach_extension(StachVersion.V2)
    tables = ext.convert(json.dumps(api_response.to_dict(), default=str))
    return [pd.DataFrame(t.data, columns=t.columns) for t in tables]

frames = []
for unit_id, res, _ in results:
    tile_name, code, basis = UNIT_KEYS[unit_id]
    s, tile_cfg = strategy_config(code), TILES[tile_name]
    for i, df in enumerate(stach_to_dataframes(res)):
        df = df.copy()
        # Provenance first, so the grain is legible without joining anything.
        # start_date is the resolved value actually sent, not the "INCEPTION" token.
        for pos, (col, val) in enumerate([
            ("asof_date",     asof_tag),
            ("tile",          tile_name),
            ("strategy_code", code),
            ("strategy",      s["label"]),
            ("fee_basis",     basis),
            ("account_id",    s["acct"]),
            ("benchmark_id",  s["benchmark"]["id"]),
            ("bench_group",   s["bench_group"]),
            ("universe_id",   s["universe"] if tile_cfg["needs_universe"] else None),
            ("start_date",    resolve_startdate(tile_cfg, code)),
            ("frequency",     tile_cfg["frequency"]),
            ("componentid",   RESOLVED_COMPONENTS[tile_name]),
            ("table_ix",      i),
        ]):
            df.insert(pos, col, val)
        frames.append(df)

tidy = pd.concat(frames, ignore_index=True)
tidy.columns = [str(c).strip().replace(" ", "_").lower() for c in tidy.columns]
tidy = tidy.astype({c: "string" for c in tidy.select_dtypes("object").columns})

print(tidy.shape)
print(tidy.groupby(["tile", "fee_basis"], dropna=False).size())
# Confirms the inception reset landed: one distinct start_date per strategy on the
# inception tiles, and a single -3Y-derived window on the risk tile.
print("\nstart_date by strategy:")
print(tidy.groupby(["strategy_code", "tile"])["start_date"].unique())

# --- ragged monthly series ------------------------------------------------
# The two monthly time-series tiles start at each strategy's OWN inception, so the four
# series are deliberately ragged — LC, SMID, LCS and CONC each begin on a different date
# and have different row counts. That is correct, not a bug to pad over.
#
# It does have a downstream consequence: a shared date dimension in the semantic model
# will happily show a period where only some strategies existed. Any cross-strategy
# aggregate over such a period is comparing a live composite against nothing. Guard it in
# DAX against each strategy's first period rather than trusting the date table, and never
# average returns across strategies over a window one of them did not span.
_ts_tiles = [t for t, c in TILES.items() if c.get("time_series")]
if _ts_tiles:
    print("\nragged monthly series — first/last period and row count per strategy:")
    _ts = tidy[tidy["tile"].isin(_ts_tiles)]
    _date_col = next((c for c in ("date", "period", "asofdate") if c in _ts.columns), None)
    if _date_col:
        print(_ts.groupby(["tile", "strategy_code"])[_date_col]
                 .agg(["min", "max", "count"]))
    else:
        # Column naming comes from the component, so it is not knowable in advance.
        print("  (no obvious date column; inspect _ts.columns and set _date_col)")
        print("  columns:", list(_ts.columns))
        print(_ts.groupby(["tile", "strategy_code"]).size())

display(tidy.head(20))

In [ ]:
# === Cell 9: idempotent write to hbcm_datahub =============================
# delete-then-append on asof_date, so a pipeline retry or manual re-run replaces the
# snapshot instead of doubling it. mode="append" alone would silently duplicate.
from deltalake import DeltaTable, write_deltalake

try:
    DeltaTable(TABLE_PATH).delete(f"asof_date = '{asof_tag}'")
    mode = "append"
except Exception:
    mode = "overwrite"          # table does not exist yet

write_deltalake(TABLE_PATH, tidy, mode=mode, schema_mode="merge")
print(f"wrote {len(tidy)} rows to factset.spar_composite_returns (mode={mode})")

# No partitioning: this table is kilobytes. Partitioning a small table costs more in
# metadata than it saves in pruning.
# Re-frame any Direct Lake semantic model BEFORE running VACUUM — vacuuming files a
# framed model still points at gives users query errors on missing files.
# Order is always: write -> frame -> vacuum.

## Validation checklist

- [ ] Cell 2's version assert passed, so the Environment really is on SDK >= 3.0.0.
- [ ] 56 units expected (7 x 4 x 2); all 7 batches succeeded and no unit failed.
- [ ] Cell 3's two printed dates agree — `0CQ` should resolve to the same quarter end as
      `AS_OF_ABS`. If the data's last period isn't that quarter, the `asof_date` label is
      lying and you want `USE_ABSOLUTE_AS_OF = True`.
- [ ] Gross exceeds net for every strategy and period. If gross == net, the two
      `returntype` values in Cell 3 aren't resolving to distinct streams — re-read Cell 4a.
- [ ] The two peer tiles carry a non-null `universe_id`; the other five carry null.
- [ ] `start_date` on inception tiles matches each composite's actual inception, and
      `-3Y` appears only on the risk tile.
- [ ] Returns tie to the composite performance report — that report, not SPAR, is the
      GIPS authority.

## Known sharp edges

| Symptom | Cause |
|---|---|
| 400, no detail | Wrong `prefix` on account or benchmark. The single most common failure. |
| 400 on dates | `0CQ` or the tile's frequency unsupported for that component — check Cell 4d, or set `USE_ABSOLUTE_AS_OF = True` to sidestep relative-token parsing entirely. |
| Data ends a quarter early or late | `0CQ` resolved to a different quarter than `AS_OF_ABS` assumed. Send the absolute date. |
| 400 mentioning `universeid` | Environment is on an SDK older than 2.1 — the field didn't exist. Cell 2's assert should catch this first. |
| `ImportError` on `spar_peer_universe_api` | Same cause: stale SDK. |
| Gross and net identical | Both `returntype` values resolving to the same stream. |
| Peer tile returns no percentiles | `universe` id wrong, or the component's saved universe conflicts with the override. |
| 404 fetching a result | Calculation id expired (TTL is hours). Resubmit that batch. |
| 429 | Concurrency ceiling, typically 5–10 concurrent calcs per user. Batching per tile is already the mitigation; don't set `BATCH_PER_TILE = False` on the full grid. |
| One tile empty, others fine | That component's saved date range or grouping conflicts with the override. Per-tile batching means it can't take the rest down. |
| Works interactively, fails in pipeline | `%pip` install — bind a Fabric Environment instead. |

## Sources

Verified against **upstream `FactSet/enterprise-sdk` `main`** (SPAREngine v3, SDK 3.0.0)
— `SPARCalculationsApi.md`, `SPARCalculationParameters.md`, `SPARIdentifier.md`,
`SPARDateParameters.md`, `CalculationMeta.md`, `AccountsApi.md`
(`get_spar_returns_type`), `SPARPeerUniverseApi.md`, `FrequenciesApi.md`,
`ReturnType.md`, plus `BREAKING.md` for the 2026-05-20 Python-SDK bump.

Not against `code/python/SPAREngine/v3/` in this repo, which is pinned at 2.0.3
(2025-07-21) and lacks `universeid` and `SPARPeerUniverseApi`.

Microsoft Learn — [notebook limitations](https://learn.microsoft.com/fabric/data-engineering/notebook-limitation),
[%run](https://learn.microsoft.com/fabric/data-engineering/author-execute-notebook#run-notebooks),
[Python kernel lifecycle](https://learn.microsoft.com/fabric/data-engineering/python-notebook-runtime-lifecycle),
[pandas to lakehouse](https://learn.microsoft.com/fabric/data-engineering/lakehouse-notebook-load-data#load-data-with-pandas-api).